In [1]:
import pandas as pd
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("chicago/chicago-food-inspections")
print(os.listdir(path))

['food-inspections.csv', 'socrata_metadata.json']


In [2]:
%pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_csv(path + "/food-inspections.csv")
df.head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,...,Results,Violations,Latitude,Longitude,Location,Historical Wards 2003-2015,Zip Codes,Community Areas,Census Tracts,Wards
0,2352683,CHIPOTLE MEXICAN GRILL,CHIPOTLE MEXICAN GRILL,2670642.0,Restaurant,Risk 1 (High),1025 W ADDISON ST,CHICAGO,IL,60613.0,...,Pass w/ Conditions,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.947118,-87.655381,"{'latitude': '-87.6553811622124', 'longitude':...",NaN,NaN,NaN,NaN,NaN
1,2352734,CHILI'S T-I,CHILI'S (T1-B14),34169.0,Restaurant,Risk 1 (High),11601 W TOUHY AVE,CHICAGO,IL,60666.0,...,Pass,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,42.008536,-87.914428,"{'latitude': '-87.91442843927047', 'longitude'...",NaN,NaN,NaN,NaN,NaN
2,2352713,NICK'S FIRE GRILL STEAK & LEMONADE INC.,NICK'S FIRE GRILL STEAK & LEMONADE INC.,2699090.0,NaN,Risk 1 (High),2900 W 63RD ST,CHICAGO,IL,60629.0,...,Not Ready,NaN,41.779190,-87.695892,"{'latitude': '-87.69589237398256', 'longitude'...",NaN,NaN,NaN,NaN,NaN
3,2352701,TAQUERIA BLUE LINE INC.,NaN,2703436.0,Restaurant,Risk 1 (High),3401 W BELMONT AVE,CHICAGO,IL,60618.0,...,Pass w/ Conditions,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.939168,-87.712546,"{'latitude': '-87.71254599117191', 'longitude'...",NaN,NaN,NaN,NaN,NaN
4,2352727,PORTAGE PARK DAY NURSERY,MOSAIC EARLY CHILDHOOD ACADEMY,2215815.0,Children's Services Facility,Risk 1 (High),5332-5334 W ADDISON ST,CHICAGO,IL,60641.0,...,Pass,NaN,41.946065,-87.760722,"{'latitude': '-87.76072227616888', 'longitude'...",NaN,NaN,NaN,NaN,NaN


In [4]:
df.shape

(196825, 22)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 196825 entries, 0 to 196824
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Inspection ID               196825 non-null  int64  
 1   DBA Name                    196825 non-null  str    
 2   AKA Name                    194367 non-null  str    
 3   License #                   196808 non-null  float64
 4   Facility Type               192057 non-null  str    
 5   Risk                        196756 non-null  str    
 6   Address                     196825 non-null  str    
 7   City                        196686 non-null  str    
 8   State                       196783 non-null  str    
 9   Zip                         196775 non-null  float64
 10  Inspection Date             196825 non-null  str    
 11  Inspection Type             196824 non-null  str    
 12  Results                     196825 non-null  str    
 13  Violations               

## Initial Data Inspection

**Dataset:** Chicago Food Inspections  
**Source:** City of Chicago (via Kaggle)  
**Original size:** 196,825 rows × 22 columns

### Observations
1. Several columns contain missing values.
2. Five geographical columns are completely empty.
3. Inspection Date is stored as text instead of datetime.
4. License # and Zip are stored as float64.
5. The Violations column contains many missing values.

### Cleaning Status
No changes made. Issues identified for further investigation.

In [6]:
df.isnull().sum()

Inspection ID                      0
DBA Name                           0
AKA Name                        2458
License #                         17
Facility Type                   4768
Risk                              69
Address                            0
City                             139
State                             42
Zip                               50
Inspection Date                    0
Inspection Type                    1
Results                            0
Violations                     52266
Latitude                         690
Longitude                        690
Location                         690
Historical Wards 2003-2015    196825
Zip Codes                     196825
Community Areas               196825
Census Tracts                 196825
Wards                         196825
dtype: int64

## Missing Values Assessment

The dataset contains 196,825 records and 22 columns.

### Findings
- Five geographic columns are completely empty.
- Violations contains 52,266 missing values.
- Facility Type contains 4,768 missing values.
- AKA Name contains 2,458 missing values.
- Latitude, Longitude, and Location each contain 690 missing values.
- Other columns contain smaller numbers of missing values.
- Inspection ID, DBA Name, Address, Inspection Date, and Results have no missing values.

### Cleaning Decision
No missing values have been modified or removed.

Further investigation is required to determine whether missing values represent unavailable information, non-applicable fields, or data-quality problems.

In [7]:
df[df["Violations"].isnull()]["Results"].value_counts()

Results
Pass                    23978
Out of Business         16890
No Entry                 5891
Fail                     3113
Not Ready                1861
Pass w/ Conditions        464
Business Not Located       69
Name: count, dtype: int64

## Investigation: Missing Violations

### Observation
The Violations column contains 52,266 missing values.

### Investigation
Used the following code to examine inspection results where violations were missing:

`df[df["Violations"].isnull()]["Results"].value_counts()`

### Findings
- Pass: 23,978
- Out of Business: 16,890
- No Entry: 5,891
- Fail: 3,113
- Not Ready: 1,861
- Pass w/ Conditions: 464
- Business Not Located: 69

### Interpretation
Missing violations occur across different inspection outcomes, including failed inspections.

Therefore, missing violations cannot automatically be interpreted as inspections with no violations.

### Cleaning Decision
Retain missing values pending further investigation.

No records have been modified or deleted.

In [8]:
df[(df["Results"] == "Fail") & (df["Violations"].isnull())].head()

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,...,Results,Violations,Latitude,Longitude,Location,Historical Wards 2003-2015,Zip Codes,Community Areas,Census Tracts,Wards
13,2352658,CHIPOTLE MEXICAN GRILL,CHIPOTLE MEXICAN GRILL,2670643.0,Restaurant,Risk 1 (High),1025 W ADDISON ST,CHICAGO,IL,60613.0,...,Fail,NaN,41.947118,-87.655381,"{'latitude': '-87.6553811622124', 'longitude':...",NaN,NaN,NaN,NaN,NaN
121,2352332,NEMA CHICAGO,NEMA CHICAGO,2621255.0,TAVERN,Risk 3 (Low),1200 S INDIANA AVE,CHICAGO,IL,60605.0,...,Fail,NaN,41.867398,-87.622701,"{'latitude': '-87.6227005115827', 'longitude':...",NaN,NaN,NaN,NaN,NaN
147,2352340,TAKITO STREET,TAKITO STREET,2658194.0,Restaurant,Risk 3 (Low),2421 - 2423 N LINCOLN AVE,CHICAGO,IL,60614.0,...,Fail,NaN,41.925997,-87.649322,"{'latitude': '-87.64932237272059', 'longitude'...",NaN,NaN,NaN,NaN,NaN
167,2352269,LOCAL MARKET,LOCAL MARKET,2689708.0,Grocery Store,Risk 3 (Low),2101 E 71ST ST,CHICAGO,IL,60649.0,...,Fail,NaN,41.765990,-87.573787,"{'latitude': '-87.57378733665412', 'longitude'...",NaN,NaN,NaN,NaN,NaN
188,2352266,LOCAL MARKET,LOCAL MARKET,2689709.0,Grocery Store,Risk 3 (Low),2101 E 71ST ST,CHICAGO,IL,60649.0,...,Fail,NaN,41.765990,-87.573787,"{'latitude': '-87.57378733665412', 'longitude'...",NaN,NaN,NaN,NaN,NaN


In [9]:
df[(df["Results"] == "Fail") & (df["Violations"].isnull())]["Inspection Type"].value_counts()

Inspection Type
License                                     1939
Canvass                                      459
Complaint                                    128
Out of Business                              112
License-Task Force                           100
License Re-Inspection                         59
Short Form Complaint                          57
Task Force Liquor 1475                        47
Consultation                                  45
No Entry                                      28
Complaint-Fire                                22
Short Form Fire-Complaint                     17
Canvass Re-Inspection                         16
Pre-License Consultation                      11
OUT OF BUSINESS                                8
Package Liquor 1474                            7
Not Ready                                      6
Complaint Re-Inspection                        6
NO ENTRY                                       6
Recent Inspection                              5
Non-

## Investigation: Inspection Types with Missing Violations

### Objective
Investigate the inspection types associated with failed inspections that have missing violation details.

### Python Code
`df[(df["Results"] == "Fail") & (df["Violations"].isnull())]["Inspection Type"].value_counts()`

### Findings

The investigation identified the inspection types associated with failed inspections that have missing violation details.

The most common inspection types were:

- License: 1,939
- Canvass: 459
- Complaint: 128
- Out of Business: 112
- License-Task Force: 100
- License Re-Inspection: 59
- Short Form Complaint: 57
- Task Force Liquor 1475: 47
- Consultation: 45
- No Entry: 28

Other inspection types account for the remaining records.

### Data Quality Issues

The Inspection Type column contains inconsistent labels, including:

- No Entry / NO ENTRY / no entry
- Out of Business / OUT OF BUSINESS

### Cleaning Decision

- Retain missing violation values.
- Standardise inconsistent inspection-type labels during cleaning.
- Avoid combining categories that may represent different inspection activities.

In [10]:
df["Inspection Type"].value_counts()

Inspection Type
Canvass                          103995
License                           26054
Canvass Re-Inspection             20722
Complaint                         18223
License Re-Inspection              9018
                                  ...  
LIQOUR TASK FORCE NOT READY           1
TASK FORCE NOT READY                  1
POSSIBLE FBI                          1
Kids Cafe'                            1
TASK FORCE PACKAGE GOODS 1474         1
Name: count, Length: 108, dtype: int64

In [11]:
df["Inspection Type"].nunique()

108

### Inspection Type Distribution

**Code:** `df["Inspection Type"].value_counts()`

**Findings:**
- 108 distinct inspection-type labels.
- Most common: Canvass (103,995), License (26,054), Canvass Re-Inspection (20,722), Complaint (18,223), and License Re-Inspection (9,018).
- Inconsistent capitalisation exists across some labels.

**Decision:** Investigate inconsistent labels before standardisation. No changes made yet.

In [12]:
df["Inspection Type"].str.strip().str.lower().nunique()

96

In [13]:
df["Inspection Type"].value_counts().tail(30)

Inspection Type
task force(1470) liquor tavern              1
LICENSE WRONG ADDRESS                       1
error save                                  1
CHANGED COURT DATE                          1
CANVASS/SPECIAL EVENT                       1
DAY CARE LICENSE RENEWAL                    1
LIQUOR CATERING                             1
Summer Feeding                              1
TASK FORCE PACKAGE LIQUOR                   1
citation re-issued                          1
TASTE OF CHICAGO                            1
HACCP QUESTIONAIRE                          1
LICENSE                                     1
out ofbusiness                              1
CLOSE-UP/COMPLAINT REINSPECTION             1
finish complaint inspection from 5-18-10    1
Duplicated                                  1
license                                     1
sfp/complaint                               1
RECALL INSPECTION                           1
Special Task Force                          1
Business Not Locat

In [14]:
df.duplicated().sum()

np.int64(149)

### Duplicate Records Investigation

**Code:** `df.duplicated().sum()`

**Finding:** Identified 149 exact duplicate rows in the dataset.

**Decision:** Investigate the duplicates before deciding whether to remove them. No changes made yet.

In [15]:
df[df.duplicated(keep=False)].head(10)

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,...,Results,Violations,Latitude,Longitude,Location,Historical Wards 2003-2015,Zip Codes,Community Areas,Census Tracts,Wards
9932,2286181,YATAI,YATAI,2641287.0,Restaurant,Risk 1 (High),6230 N BROADWAY,CHICAGO,IL,60660.0,...,Pass,37. FOOD PROPERLY LABELED; ORIGINAL CONTAINER ...,41.995461,-87.660580,"{'latitude': '-87.66057971349507', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9935,2286112,PILSEN BREAKFAST & GRILL,PILSEN BREAKFAST & GRILL,2340664.0,Restaurant,Risk 1 (High),1454 W 18TH ST,CHICAGO,IL,60608.0,...,Out of Business,NaN,41.857947,-87.663537,"{'latitude': '-87.66353726681123', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9944,2286143,CABANA BAR,CABANA BAR,2636260.0,Liquor,Risk 3 (Low),5342 N CUMBERLAND AVE,CHICAGO,IL,60656.0,...,Fail,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.976805,-87.836761,"{'latitude': '-87.83676096082264', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9945,2286108,DRINK SHOP,DRINK SHOP,2630762.0,TAVERN,Risk 3 (Low),2513 W ARMITAGE AVE,CHICAGO,IL,60647.0,...,Pass w/ Conditions,5. PROCEDURES FOR RESPONDING TO VOMITING AND D...,41.917439,-87.690475,"{'latitude': '-87.69047489099341', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9946,2286137,"CERES FOOD GROUP,INC.",(SCHOOL) PERSPECTIVIES IIT MATH & SCIENCE,3709480.0,School,Risk 1 (High),3663 S WABASH AVE,CHICAGO,IL,60653.0,...,Fail,50. HOT & COLD WATER AVAILABLE; ADEQUATE PRESS...,41.827796,-87.624751,"{'latitude': '-87.62475128513913', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9947,2286174,BLUE SPOT SUSHI CORP.,BLUE SPOT SUSHI CORP.,2609033.0,Restaurant,Risk 1 (High),131 N CLINTON ST,CHICAGO,IL,60661.0,...,Fail,2. CITY OF CHICAGO FOOD SERVICE SANITATION CER...,41.884188,-87.641120,"{'latitude': '-87.64111966683218', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9948,2286141,LAURA'S PIZZA AND PASTA INC.,LAURA'S PIZZA AND PASTA INC.,2663729.0,Restaurant,Risk 1 (High),1546 W 35TH ST,CHICAGO,IL,60609.0,...,Pass w/ Conditions,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.830619,-87.664981,"{'latitude': '-87.66498111376346', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9949,2286145,LITTLE NEST CARE INC.,LITTLE NEST CARE INC.,2492597.0,Children's Services Facility,Risk 1 (High),5426 W DEVON AVE,CHICAGO,IL,60646.0,...,Pass,NaN,41.997442,-87.764859,"{'latitude': '-87.76485895075076', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9972,2286173,TOCCO,TOCCO (T5 M-07),2192968.0,Restaurant,Risk 1 (High),11601 W TOUHY AVE,CHICAGO,IL,60666.0,...,Fail,5. PROCEDURES FOR RESPONDING TO VOMITING AND D...,42.008536,-87.914428,"{'latitude': '-87.91442843927047', 'longitude'...",NaN,NaN,NaN,NaN,NaN
9977,2286157,ONE STOP FOOD & GROCERY,ONE STOP FOOD & GROCERY,2027100.0,Grocery Store,Risk 2 (Medium),7357 S RACINE AVE,CHICAGO,IL,60636.0,...,Fail,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.759704,-87.653832,"{'latitude': '-87.65383153233266', 'longitude'...",NaN,NaN,NaN,NaN,NaN


In [16]:
df["Inspection ID"].duplicated().sum()

np.int64(149)

### Duplicate Records Investigation

**Code used:**
```python
df.duplicated().sum()
df["Inspection ID"].duplicated().sum()
```

**Findings:**
- 149 exact duplicate rows.
- 149 repeated Inspection IDs.

**Decision:** Remove exact duplicates during cleaning, retaining one record per inspection. No changes made yet.

In [17]:
df["Results"].value_counts()

Results
Pass                    106066
Fail                     38087
Pass w/ Conditions       27448
Out of Business          16919
No Entry                  6324
Not Ready                 1912
Business Not Located        69
Name: count, dtype: int64

In [18]:
df["Risk"].value_counts(dropna=False)

Risk
Risk 1 (High)      141129
Risk 2 (Medium)     38431
Risk 3 (Low)        17165
NaN                    69
All                    31
Name: count, dtype: int64

### Risk Category Investigation

**Code:** `df["Risk"].value_counts(dropna=False)`

**Findings:**
- High Risk (141,129) is the most common category.
- 69 records have missing risk values.
- 31 records contain "All", requiring further investigation.

**Decision:** Investigate "All" and missing risk values before cleaning.

In [19]:
df[df["Risk"] == "All"].head(10)

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,...,Results,Violations,Latitude,Longitude,Location,Historical Wards 2003-2015,Zip Codes,Community Areas,Census Tracts,Wards
20,2352633,THE SUNDAE STOP,THE SUNDAE STOP,2703510.0,NaN,All,931 W BELMONT AVE,CHICAGO,IL,60657.0,...,Not Ready,NaN,41.939820,-87.652748,"{'latitude': '-87.6527475003377', 'longitude':...",NaN,NaN,NaN,NaN,NaN
80,2352462,"CIBO FOODS,LLC","CIBO FOODS,LLC",2703767.0,Restaurant,All,1258 W JACKSON BLVD,CHICAGO,IL,60607.0,...,Not Ready,NaN,41.877861,-87.659332,"{'latitude': '-87.65933207232133', 'longitude'...",NaN,NaN,NaN,NaN,NaN
135,2352596,PREPPED N PLATED LLC,PREPPED N PLATED,2671690.0,NaN,All,3430 N SOUTHPORT AVE,CHICAGO,IL,60657.0,...,Not Ready,NaN,41.944375,-87.664091,"{'latitude': '-87.66409051548939', 'longitude'...",NaN,NaN,NaN,NaN,NaN
1960,2315596,TWO HEARTED QUEEN,TWO HEARTED QUEEN,2694759.0,Restaurant,All,3500 N HALSTED ST,CHICAGO,IL,60657.0,...,Not Ready,NaN,41.945504,-87.649579,"{'latitude': '-87.64957851411022', 'longitude'...",NaN,NaN,NaN,NaN,NaN
2177,2315299,KILWINS ICE CREAM AND CHOCOLATE,KILWINS ICE CREAM AND CHOCOLATE,2694404.0,NaN,All,3519 N CLARK ST,CHICAGO,IL,60657.0,...,Not Ready,NaN,41.946281,-87.655603,"{'latitude': '-87.65560312808297', 'longitude'...",NaN,NaN,NaN,NaN,NaN
2750,2312632,SUPERMERCADO EL PARAISO INC.,SUPERMERCADO EL PARAISO INC.,2670414.0,Grocery Store,All,4301 S ARCHER AVE,CHICAGO,IL,60632.0,...,Out of Business,NaN,41.815301,-87.702042,"{'latitude': '-87.70204160078063', 'longitude'...",NaN,NaN,NaN,NaN,NaN
2999,2312348,KHEPRI CAFE,KHEPRI CAFE,2683983.0,NaN,All,4650 N KEDZIE AVE,CHICAGO,IL,60625.0,...,Not Ready,NaN,41.966062,-87.708548,"{'latitude': '-87.7085480707044', 'longitude':...",NaN,NaN,NaN,NaN,NaN
5827,2303377,TACO BELL #35850,TACO BELL,2665149.0,Restaurant,All,22 E CHICAGO AVE,CHICAGO,IL,60611.0,...,Out of Business,NaN,41.896799,-87.627266,"{'latitude': '-87.62726628946932', 'longitude'...",NaN,NaN,NaN,NaN,NaN
5880,2303270,SIPPING TURTLE CAFE,NaN,2677835.0,NaN,All,2959 N MILWAUKEE AVE,CHICAGO,IL,60618.0,...,Not Ready,NaN,41.935107,-87.717072,"{'latitude': '-87.7170721521948', 'longitude':...",NaN,NaN,NaN,NaN,NaN
6727,2300322,TACO BELL #35850,TACO BELL,2665149.0,Restaurant,All,22 E CHICAGO AVE,CHICAGO,IL,60611.0,...,Not Ready,NaN,41.896799,-87.627266,"{'latitude': '-87.62726628946932', 'longitude'...",NaN,NaN,NaN,NaN,NaN


In [20]:
df[df["Risk"] == "All"]["Results"].value_counts()

Results
Not Ready               17
Out of Business          5
Fail                     5
No Entry                 3
Business Not Located     1
Name: count, dtype: int64

### Risk Category Investigation

**Code:**
```python
df["Risk"].value_counts(dropna=False)
df[df["Risk"] == "All"]["Results"].value_counts()
```

**Findings:**
- High Risk is the most common category (141,129).
- 69 records have missing risk values.
- 31 records are classified as "All", with none receiving a Pass or Pass w/ Conditions result.

**Decision:** Retain "All" and missing risk values pending further investigation.

In [21]:
df["Inspection Date"].head(10)

0    2019-12-04T00:00:00.000
1    2019-12-04T00:00:00.000
2    2019-12-04T00:00:00.000
3    2019-12-04T00:00:00.000
4    2019-12-04T00:00:00.000
5    2019-12-04T00:00:00.000
6    2019-12-04T00:00:00.000
7    2019-12-04T00:00:00.000
8    2019-12-04T00:00:00.000
9    2019-12-04T00:00:00.000
Name: Inspection Date, dtype: str

In [22]:
pd.to_datetime(df["Inspection Date"], errors="coerce").isna().sum()

np.int64(0)

In [23]:
dates = pd.to_datetime(df["Inspection Date"])

print("Earliest Date:", dates.min())
print("Latest Date:", dates.max())

Earliest Date: 2010-01-04 00:00:00
Latest Date: 2019-12-04 00:00:00


### Inspection Date Investigation

**Findings:**
- Date range: January 4, 2010 – December 4, 2019.
- No missing or invalid dates.
- Dates are stored as text rather than datetime.

**Decision:** Convert the Inspection Date column to datetime during cleaning. No changes made yet.

In [24]:
df["Facility Type"].value_counts().head(15)

Facility Type
Restaurant                         131254
Grocery Store                       25034
School                              12132
Children's Services Facility         3115
Bakery                               2884
Daycare (2 - 6 Years)                2703
Daycare Above and Under 2 Years      2400
Long Term Care                       1352
Catering                             1201
Mobile Food Dispenser                 868
Liquor                                861
Daycare Combo 1586                    758
Mobile Food Preparer                  639
Golden Diner                          569
Hospital                              557
Name: count, dtype: int64

### Facility Type Investigation

**Code:** `df["Facility Type"].value_counts().head(15)`

**Findings:**
- Restaurants (131,254) have the highest number of inspection records, followed by Grocery Stores (25,034) and Schools (12,132).
- 4,768 records have missing facility types.
- Further checks are needed to identify inconsistent category labels.

**Decision:** Review category consistency before cleaning.

In [25]:
df["Facility Type"].nunique()

491

In [26]:
df["Facility Type"].str.strip().str.lower().nunique()

442

### Facility Type Investigation

**Findings:**
- Restaurants (131,254), Grocery Stores (25,034), and Schools (12,132) are the most frequently inspected facility types.
- 4,768 missing values.
- 491 unique labels, reduced to 442 after standardising spacing and capitalisation.

**Decision:** Standardise formatting during cleaning while preserving distinct facility categories.

In [27]:
df["City"].value_counts(dropna=False).head(10)

City
CHICAGO              195986
Chicago                 320
NaN                     139
chicago                  97
CCHICAGO                 46
SCHAUMBURG               25
MAYWOOD                  15
ELK GROVE VILLAGE        13
CHicago                  12
EVANSTON                 11
Name: count, dtype: int64

### City Investigation

**Findings:**
- Chicago accounts for most inspection records.
- Inconsistent capitalisation and possible spelling errors identified.
- 139 missing city values.
- Other city names are also present.

**Decision:** Standardise city names during cleaning and investigate possible spelling errors.

In [28]:
df["City"].str.upper().value_counts(dropna=False).head(15)

City
CHICAGO              196415
NaN                     139
CCHICAGO                 46
SCHAUMBURG               25
MAYWOOD                  16
ELK GROVE VILLAGE        13
EVANSTON                 11
CHESTNUT STREET          10
CICERO                    9
SKOKIE                    8
INACTIVE                  8
NILES NILES               7
CHICAGOCHICAGO            7
ELMHURST                  6
CALUMET CITY              6
Name: count, dtype: int64

### City Investigation

**Findings:**
- Chicago accounts for 196,415 records after standardising capitalisation.
- 139 missing values.
- Inconsistent capitalisation, possible spelling errors, and suspicious city entries identified.

**Decision:** Standardise city names and review suspicious entries during cleaning. Preserve valid locations.

In [29]:
df["State"].value_counts(dropna=False)

State
IL     196780
NaN        42
IN          1
NY          1
WI          1
Name: count, dtype: int64

### State Investigation

**Findings:**
- Illinois accounts for 196,780 records.
- 42 missing values and 3 unexpected state codes (IN, NY, WI).

**Decision:** Verify unusual state entries against location details during cleaning. No changes made yet.

In [30]:
df["Zip"].head(10)

0    60613.0
1    60666.0
2    60629.0
3    60618.0
4    60641.0
5    60605.0
6    60613.0
7    60657.0
8    60659.0
9    60641.0
Name: Zip, dtype: float64

### ZIP Code Investigation

**Findings:**
- ZIP codes are stored as `float64` instead of text.
- 50 missing values.
- Sample values contain unnecessary decimal points.

**Decision:** Convert ZIP codes to text during cleaning, preserving missing values.

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 196825 entries, 0 to 196824
Data columns (total 22 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Inspection ID               196825 non-null  int64  
 1   DBA Name                    196825 non-null  str    
 2   AKA Name                    194367 non-null  str    
 3   License #                   196808 non-null  float64
 4   Facility Type               192057 non-null  str    
 5   Risk                        196756 non-null  str    
 6   Address                     196825 non-null  str    
 7   City                        196686 non-null  str    
 8   State                       196783 non-null  str    
 9   Zip                         196775 non-null  float64
 10  Inspection Date             196825 non-null  str    
 11  Inspection Type             196824 non-null  str    
 12  Results                     196825 non-null  str    
 13  Violations               

In [32]:
df["License #"].head(10)

0    2670642.0
1      34169.0
2    2699090.0
3    2703436.0
4    2215815.0
5    1801618.0
6    2670643.0
7    2698776.0
8      23931.0
9    2506383.0
Name: License #, dtype: float64

### License Number Investigation

**Findings:**
- License numbers are stored as `float64`.
- 17 missing values.
- Sample values contain unnecessary decimal points.

**Decision:** Convert license numbers to text during cleaning, preserving missing values.

In [33]:
df[["Latitude", "Longitude"]].describe()

,Latitude,Longitude
count,196135.000000,196135.000000
mean,41.881231,-87.676819
std,0.080846,0.058969
min,41.644670,-87.914428
25%,41.834897,-87.707742
50%,41.891980,-87.666812
75%,41.939814,-87.634955
max,42.021064,-87.525094


### Geographic Coordinates Investigation

**Findings:**
- Latitude and Longitude each contain 196,135 valid numeric entries and 690 missing values.
- Coordinate ranges appear geographically reasonable.

**Decision:** Retain both columns for Power BI mapping. Preserve missing coordinates.

In [34]:
df["Location"].head(10)

0    {'latitude': '-87.6553811622124', 'longitude':...
1    {'latitude': '-87.91442843927047', 'longitude'...
2    {'latitude': '-87.69589237398256', 'longitude'...
3    {'latitude': '-87.71254599117191', 'longitude'...
4    {'latitude': '-87.76072227616888', 'longitude'...
5    {'latitude': '-87.62935653990546', 'longitude'...
6    {'latitude': '-87.6553811622124', 'longitude':...
7    {'latitude': '-87.65299525711882', 'longitude'...
8    {'latitude': '-87.70056525028134', 'longitude'...
9    {'latitude': '-87.72750513009342', 'longitude'...
Name: Location, dtype: str

In [35]:
print(df.loc[0, ["Latitude", "Longitude", "Location"]])

Latitude                                             41.947118
Longitude                                           -87.655381
Location     {'latitude': '-87.6553811622124', 'longitude':...
Name: 0, dtype: object


In [36]:
print(df.loc[0, "Location"])

{'latitude': '-87.6553811622124', 'longitude': '41.94711774835235'}


### Geographic Coordinates Investigation

**Findings:**
- Latitude, Longitude, and Location each contain 690 missing values.
- Numeric coordinate ranges appear reasonable.
- The Location column stores coordinates as text, with reversed latitude/longitude labels identified in a sample record.

**Decision:** Retain the numeric Latitude and Longitude columns for mapping. Remove the redundant Location column during cleaning.

## Data Quality Assessment Summary

**Dataset:** Chicago Food Inspections (2010–2019)  
**Original size:** 196,825 rows × 22 columns

| Area | Findings | Cleaning Decision |
|---|---|---|
| Duplicates | 149 exact duplicate records and repeated Inspection IDs | Remove duplicate copies |
| Missing Violations | 52,266 missing, including 3,113 failed inspections | Preserve missing values |
| Inspection Type | 108 unique labels; 96 after formatting standardisation | Standardise capitalisation and spacing |
| Facility Type | 4,768 missing; 491 unique labels reduced to 442 after formatting checks | Standardise formatting; preserve valid categories |
| Results | Seven consistent inspection outcomes | Retain all categories |
| Risk | 69 missing; 31 labelled "All" | Retain and flag unresolved values |
| Inspection Date | Text format; no invalid dates; 2010–2019 | Convert to datetime |
| City | 139 missing; inconsistent capitalisation and suspicious entries | Standardise and correct verified errors |
| State | 42 missing; three unexpected state codes | Verify unusual entries |
| ZIP & License Numbers | Stored as `float64`; 50 and 17 missing respectively | Convert identifiers to text |
| Geographic Coordinates | 690 missing; Location contains reversed coordinate labels in an examined record | Retain numeric Latitude/Longitude; remove redundant Location |
| Five Empty Columns | All 196,825 values missing in each | Remove all five columns |

### Cleaning Strategy

- Preserve the raw dataset.
- Remove duplicate records and columns containing no usable information.
- Correct data types and standardise inconsistent labels.
- Retain missing values where replacements cannot be justified.
- Validate the cleaned dataset before exploratory data analysis (EDA).

**Status:** Initial data-quality assessment completed. Cleaning has not yet begun.

## Phase 2: Data Cleaning

In [37]:
df_clean = df.copy()

In [38]:
df_clean.shape

(196825, 22)

In [39]:
df_clean = df_clean.drop_duplicates()

In [40]:
df_clean.shape

(196676, 22)

### 1. Duplicate Removal

**Code:** `df_clean = df_clean.drop_duplicates()`

**Result:** Removed 149 exact duplicate records, reducing the dataset from 196,825 to 196,676 rows.

**Reason:** Prevent duplicate inspections from inflating analysis results.

In [41]:
df_clean = df_clean.dropna(axis=1, how="all")

In [42]:
df_clean.shape

(196676, 17)

### 2. Removing Empty Columns

**Code:** `df_clean = df_clean.dropna(axis=1, how="all")`

**Result:** Removed five completely empty columns, reducing the dataset from 22 to 17 columns.

**Reason:** These columns contained no usable information.

In [43]:
df_clean = df_clean.drop(columns=["Location"])

In [44]:
df_clean.shape

(196676, 16)

### 3. Removing the Redundant Location Column

**Code:** `df_clean = df_clean.drop(columns=["Location"])`

**Result:** Removed the Location column, leaving 196,676 rows and 16 columns.

**Reason:** Latitude and Longitude already provide the geographic information needed for mapping.

In [45]:
df_clean["Inspection Date"] = pd.to_datetime(df["Inspection Date"])

In [46]:
df_clean["Inspection Date"].dtype 

dtype('<M8[us]')

### 4. Converting Inspection Dates

**Code:** `df_clean["Inspection Date"] = pd.to_datetime(df_clean["Inspection Date"])`

**Result:** Successfully converted Inspection Date from text to datetime.

**Reason:** Enables accurate date-based analysis and reporting.

In [47]:
df_clean["Inspection Type"] = df_clean["Inspection Type"].str.strip().str.lower()

In [48]:
df_clean["Inspection Type"].nunique()

96

### 5. Standardising Inspection Type

**Code:** `df_clean["Inspection Type"] = df_clean["Inspection Type"].str.strip().str.lower()`

**Result:** Standardised capitalisation and spacing, leaving 96 unique inspection-type labels.

**Reason:** Prevent formatting differences from creating duplicate categories.

In [49]:
df_clean["Facility Type"] = df_clean["Facility Type"].str.strip().str.lower()

In [50]:
df_clean["Facility Type"].nunique()

442

### 6. Standardising Facility Type

**Code:** `df_clean["Facility Type"] = df_clean["Facility Type"].str.strip().str.lower()`

**Result:** Reduced unique facility-type labels from 491 to 442 by standardising capitalisation and spacing.

**Reason:** Prevent formatting inconsistencies from creating duplicate categories.

In [51]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 196676 entries, 0 to 196824
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Inspection ID    196676 non-null  int64         
 1   DBA Name         196676 non-null  str           
 2   AKA Name         194219 non-null  str           
 3   License #        196659 non-null  float64       
 4   Facility Type    191908 non-null  str           
 5   Risk             196607 non-null  str           
 6   Address          196676 non-null  str           
 7   City             196537 non-null  str           
 8   State            196634 non-null  str           
 9   Zip              196626 non-null  float64       
 10  Inspection Date  196676 non-null  datetime64[us]
 11  Inspection Type  196675 non-null  str           
 12  Results          196676 non-null  str           
 13  Violations       144442 non-null  str           
 14  Latitude         195987 non-null  fl

In [52]:
df_clean["City"] = df_clean["City"].str.strip().str.upper()

In [53]:
df_clean["City"].value_counts(dropna=False).head(15)

City
CHICAGO              196266
NaN                     139
CCHICAGO                 46
SCHAUMBURG               25
MAYWOOD                  16
ELK GROVE VILLAGE        13
EVANSTON                 11
CHESTNUT STREET          10
CICERO                    9
SKOKIE                    8
INACTIVE                  8
NILES NILES               7
CHICAGOCHICAGO            7
ELMHURST                  6
CALUMET CITY              6
Name: count, dtype: int64

### 7. Standardising City Names

**Code:** `df_clean["City"] = df_clean["City"].str.strip().str.upper()`

**Result:** Standardised city names to uppercase. CHICAGO has 196,266 records after duplicate removal.

**Outstanding issues:** Possible spelling errors and invalid city entries require verification.

In [54]:
df_clean[df_clean["City"].isin(["CCHICAGO", "CHICAGOCHICAGO"])][["City", "Address", "Zip"]].head(10)

,City,Address,Zip
7729,CCHICAGO,1127 N STATE ST,60610.0
9225,CCHICAGO,1127 N STATE ST,60610.0
9804,CHICAGOCHICAGO,505 N ST CLAIR ST,60611.0
10153,CHICAGOCHICAGO,505 N ST CLAIR ST,60611.0
21355,CHICAGOCHICAGO,2410 S WESTERN AVE,60608.0
21375,CCHICAGO,3640 N CENTRAL AVE,60634.0
23783,CCHICAGO,1127 N STATE ST,60610.0
24801,CHICAGOCHICAGO,2410 S WESTERN AVE,60608.0
32632,CCHICAGO,2454-2458 W MONTROSE AVE,60618.0
34357,CCHICAGO,2454-2458 W MONTROSE AVE,60618.0


In [55]:
df_clean[df_clean["City"].isin(["CCHICAGO", "CHICAGOCHICAGO"])]["Zip"].value_counts(dropna=False)

Zip
60634.0    14
60618.0    14
60610.0    10
60642.0     5
60614.0     3
60611.0     2
60608.0     2
60637.0     2
60612.0     1
Name: count, dtype: int64

In [56]:
df_clean["City"] = df_clean["City"].replace({"CCHICAGO": "CHICAGO", "CHICAGOCHICAGO": "CHICAGO"})

In [57]:
df_clean["City"].value_counts(dropna=False).head(10)

City
CHICAGO              196319
NaN                     139
SCHAUMBURG               25
MAYWOOD                  16
ELK GROVE VILLAGE        13
EVANSTON                 11
CHESTNUT STREET          10
CICERO                    9
SKOKIE                    8
INACTIVE                  8
Name: count, dtype: int64

### 7. Standardising City Names

**Actions:**
- Standardised capitalisation and removed unnecessary spaces.
- Corrected `CCHICAGO` and `CHICAGOCHICAGO` to `CHICAGO`.

**Result:** Corrected 53 city entries, bringing Chicago's total to 196,319 records.

**Decision:** Retain missing values and investigate other suspicious city entries before making further corrections.

In [58]:
records = df_clean[df_clean["City"] == "CHESTNUT STREET"]
records[["Address", "Zip", "State"]]

,Address,Zip,State
9589,1420 N Hudson (432W),60610.0,IL
29673,1420 N Hudson (432W),60610.0,IL
48173,1420 N Hudson (432W),60610.0,IL
75456,1420 N Hudson (432W),60610.0,IL
97673,1420 N Hudson (432W),60610.0,IL
115548,1420 N Hudson (432W),60610.0,IL
132672,1420 N Hudson (432W),60610.0,IL
151903,1420 N Hudson (432W),60610.0,IL
170688,1420 N Hudson (432W),60610.0,IL
195699,1420 N Hudson (432W),60610.0,IL


In [59]:
records = df_clean[df_clean["City"] == "CHICAGO"]
records[["Address", "Zip", "State"]].head(10)

,Address,Zip,State
0,1025 W ADDISON ST,60613.0,IL
1,11601 W TOUHY AVE,60666.0,IL
2,2900 W 63RD ST,60629.0,IL
3,3401 W BELMONT AVE,60618.0,IL
4,5332-5334 W ADDISON ST,60641.0,IL
5,600 S DEARBORN ST,60605.0,IL
6,1025 W ADDISON ST,60613.0,IL
7,3349 N CLARK ST,60657.0,IL
8,5650 N MOZART,60659.0,IL
9,4041 N PULASKI RD,60641.0,IL


In [60]:
df_clean["City"] = df_clean["City"].replace("CHESTNUT STREET", "CHICAGO")

In [61]:
df_clean["City"].value_counts().head(15)

City
CHICAGO              196329
SCHAUMBURG               25
MAYWOOD                  16
ELK GROVE VILLAGE        13
EVANSTON                 11
CICERO                    9
SKOKIE                    8
INACTIVE                  8
NILES NILES               7
ELMHURST                  6
CALUMET CITY              6
CHCHICAGO                 6
OAK PARK                  5
WORTH                     5
CHARLES A HAYES           4
Name: count, dtype: int64

### 8. Standardising City Entries

**Code:** `df_clean["City"] = df_clean["City"].replace("CHESTNUT STREET", "CHICAGO")`

**Result:** Standardised 10 city entries from CHESTNUT STREET to CHICAGO, increasing Chicago's total to 196,329 records.

**Reason:** Address and ZIP code verification confirmed that the affected records belong to Chicago, improving geographic data accuracy.

In [62]:
cities = df_clean["City"].value_counts(dropna=False)
print(cities.to_string())

City
CHICAGO                 196329
NaN                        139
SCHAUMBURG                  25
MAYWOOD                     16
ELK GROVE VILLAGE           13
EVANSTON                    11
CICERO                       9
SKOKIE                       8
INACTIVE                     8
NILES NILES                  7
ELMHURST                     6
CALUMET CITY                 6
CHCHICAGO                    6
OAK PARK                     5
WORTH                        5
CHARLES A HAYES              4
SUMMIT                       4
ALSIP                        4
ROSEMONT                     3
PLAINFIELD                   3
BRIDGEVIEW                   3
SCHILLER PARK                3
CHCICAGO                     3
CHICAGOI                     3
EAST HAZEL CREST             3
CHICAGO.                     2
CHICAGOHICAGO                2
HIGHLAND PARK                2
312CHICAGO                   2
BEDFORD PARK                 2
BANNOCKBURNDEERFIELD         2
CHICAGO HEIGHTS              2
STR

In [63]:
suspect = [
    "CHCHICAGO", "CHCICAGO", "CHICAGOI",
    "CHICAGO.", "CHICAGOHICAGO", "312CHICAGO",
    "NILES NILES", "OOLYMPIA FIELDS",
    "BANNOCKBURNDEERFIELD", "INACTIVE",
    "CHARLES A HAYES"
]

records = df_clean[df_clean["City"].isin(suspect)]

records[["City", "Address", "Zip", "State"]]

,City,Address,Zip,State
7930,NILES NILES,239 GOLF MILL CENTER STE 322,60714.0,IL
8091,CHICAGO.,6852 W ARCHER AVE,60638.0,IL
14043,CHICAGOHICAGO,1300 W CERMAK RD,60608.0,IL
23108,CHICAGO.,6852 W ARCHER AVE,60638.0,IL
24890,NILES NILES,239 GOLF MILL CENTER STE 322,60714.0,IL
25008,NILES NILES,239 GOLF MILL CENTER STE 322,60714.0,IL
25801,CHICAGOHICAGO,1300 W CERMAK RD,60608.0,IL
40509,NILES NILES,239 GOLF MILL CENTER STE 322,60714.0,IL
45521,312CHICAGO,200 S WACKER DR,60606.0,IL
46209,312CHICAGO,200 S WACKER DR,60606.0,IL


In [64]:
fixes = {
    "CHCHICAGO": "CHICAGO",
    "CHCICAGO": "CHICAGO",
    "CHICAGOI": "CHICAGO",
    "CHICAGO.": "CHICAGO",
    "CHICAGOHICAGO": "CHICAGO",
    "312CHICAGO": "CHICAGO",
    "INACTIVE": "CHICAGO",
    "CHARLES A HAYES": "CHICAGO",
    "NILES NILES": "NILES",
    "OOLYMPIA FIELDS": "OLYMPIA FIELDS",
    "BANNOCKBURNDEERFIELD": "BANNOCKBURN"
}

df_clean["City"] = df_clean["City"].replace(fixes)

In [65]:
df_clean["City"].value_counts().head(10)

City
CHICAGO              196359
SCHAUMBURG               25
MAYWOOD                  16
ELK GROVE VILLAGE        13
EVANSTON                 11
CICERO                    9
SKOKIE                    8
NILES                     7
ELMHURST                  6
CALUMET CITY              6
Name: count, dtype: int64

### 9. Correcting Inconsistent City Names

**Code:** `df_clean["City"] = df_clean["City"].replace(fixes)`

**Result:** Corrected 40 city entries using a dictionary. Chicago's count increased to 196,359 records.

**Reason:** Improve geographic accuracy by correcting verified spelling errors and invalid city entries. Missing values were preserved.

In [66]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 196676 entries, 0 to 196824
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Inspection ID    196676 non-null  int64         
 1   DBA Name         196676 non-null  str           
 2   AKA Name         194219 non-null  str           
 3   License #        196659 non-null  float64       
 4   Facility Type    191908 non-null  str           
 5   Risk             196607 non-null  str           
 6   Address          196676 non-null  str           
 7   City             196537 non-null  str           
 8   State            196634 non-null  str           
 9   Zip              196626 non-null  float64       
 10  Inspection Date  196676 non-null  datetime64[us]
 11  Inspection Type  196675 non-null  str           
 12  Results          196676 non-null  str           
 13  Violations       144442 non-null  str           
 14  Latitude         195987 non-null  fl

In [67]:
states = df_clean["State"].value_counts(dropna=False)
print(states.to_string())

State
IL     196631
NaN        42
IN          1
NY          1
WI          1


In [68]:
records = df_clean[df_clean["State"].isin(["IN", "NY", "WI"])]

records[["City", "Address", "Zip", "State"]]

,City,Address,Zip,State
5966,GRIFFITH,2011 N GRIFFIN BLVD,46319.0,IN
6675,NEW YORK,65 CARMINE ST,10014.0,NY
7320,NEW HOLSTEIN,N2660 HAYTON RD,53061.0,WI


### 10. Validating State Values

**Code:** `df_clean[df_clean["State"].isin(["IN", "NY", "WI"])]`

**Result:** Verified three unusual state entries against their cities and ZIP codes. All three were consistent.

**Reason:** Preserve valid location information rather than incorrectly replacing every state with Illinois.

In [69]:
df_clean["Zip"] = df_clean["Zip"].astype("Int64").astype("string")

In [70]:
df_clean["Zip"].head(10)

0    60613
1    60666
2    60629
3    60618
4    60641
5    60605
6    60613
7    60657
8    60659
9    60641
Name: Zip, dtype: string

### 11. Converting ZIP Codes to Text

**Code:** `df_clean["Zip"] = df_clean["Zip"].astype("Int64").astype("string")`

**Result:** Converted ZIP codes from `float64` to `string`, removing unnecessary decimal points.

**Reason:** ZIP codes are location identifiers, not numerical measurements.

In [71]:
df_clean["License #"] = df_clean["License #"].astype("Int64").astype("string")

In [72]:
df_clean["License #"].head(10)

0    2670642
1      34169
2    2699090
3    2703436
4    2215815
5    1801618
6    2670643
7    2698776
8      23931
9    2506383
Name: License #, dtype: string

### 12. Converting License Numbers to Text

**Code:** `df_clean["License #"] = df_clean["License #"].astype("Int64").astype("string")`

**Result:** Converted license numbers from `float64` to `string`, removing unnecessary decimal points.

**Reason:** License numbers are identifiers, not numerical measurements.

In [73]:
df_clean.isna().sum()

Inspection ID          0
DBA Name               0
AKA Name            2457
License #             17
Facility Type       4768
Risk                  69
Address                0
City                 139
State                 42
Zip                   50
Inspection Date        0
Inspection Type        1
Results                0
Violations         52234
Latitude             689
Longitude            689
dtype: int64

### 13. Handling Missing Values

**Code:** `df_clean.isna().sum()`

**Result:** Identified remaining missing values, including 52,234 in Violations, 4,768 in Facility Type, and 689 each in Latitude and Longitude.

**Reason:** Preserved missing values where reliable replacements were unavailable, preventing inaccurate assumptions and misleading analysis.

In [74]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 196676 entries, 0 to 196824
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Inspection ID    196676 non-null  int64         
 1   DBA Name         196676 non-null  str           
 2   AKA Name         194219 non-null  str           
 3   License #        196659 non-null  string        
 4   Facility Type    191908 non-null  str           
 5   Risk             196607 non-null  str           
 6   Address          196676 non-null  str           
 7   City             196537 non-null  str           
 8   State            196634 non-null  str           
 9   Zip              196626 non-null  string        
 10  Inspection Date  196676 non-null  datetime64[us]
 11  Inspection Type  196675 non-null  str           
 12  Results          196676 non-null  str           
 13  Violations       144442 non-null  str           
 14  Latitude         195987 non-null  fl

In [75]:
df_clean = df_clean.reset_index(drop=True)

In [76]:
df_clean.index

RangeIndex(start=0, stop=196676, step=1)

### 14. Final Data Validation

**Code:** `df_clean = df_clean.reset_index(drop=True)`

**Result:** Reset the index to 0–196,675. The cleaned dataset contains 196,676 records and 16 columns.

**Reason:** Ensure consistent row numbering after removing duplicates.

In [77]:
df_clean.to_csv("Chicago_Food_Inspections_Cleaned.csv", index=False)

### 15. Exporting the Cleaned Dataset

**Code:** `df_clean.to_csv("Chicago_Food_Inspections_Cleaned.csv", index=False)`

**Result:** Exported the cleaned dataset containing 196,676 records and 16 columns.

**Reason:** Prepare the dataset for Power BI analysis and portfolio publication.

## Phase 3: Exploratory Data Analysis (EDA)

In [78]:
df_clean["Results"].value_counts()

Results
Pass                    106003
Fail                     38055
Pass w/ Conditions       27412
Out of Business          16909
No Entry                  6322
Not Ready                 1907
Business Not Located        68
Name: count, dtype: int64

### 1. Overall Inspection Outcomes

**Code:** `df_clean["Results"].value_counts()`

**Result:** Of 196,676 records, 106,003 passed, 38,055 failed, and 27,412 passed with conditions.

**Insight:** Pass is the most common outcome. Other statuses, such as No Entry and Not Ready, must be excluded when calculating completed-inspection failure rates.

In [79]:
completed = df_clean[df_clean["Results"].isin(
    ["Pass", "Fail", "Pass w/ Conditions"]
)]

completed.shape

(171470, 16)

In [80]:
df_clean["Results"].head(10)

0    Pass w/ Conditions
1                  Pass
2             Not Ready
3    Pass w/ Conditions
4                  Pass
5                  Pass
6    Pass w/ Conditions
7    Pass w/ Conditions
8                  Pass
9                  Pass
Name: Results, dtype: str

In [81]:
completed["Results"].value_counts(normalize=True) * 100

Results
Pass                  61.820143
Fail                  22.193387
Pass w/ Conditions    15.986470
Name: proportion, dtype: float64

### 2. Completed Inspections and Failure Rate

**Code:** `completed["Results"].value_counts(normalize=True) * 100`

**Result:** Of 171,470 completed inspections, 61.82% passed, 22.19% failed, and 15.99% passed with conditions.

**Insight:** Approximately 1 in 5 completed inspections failed, highlighting the importance of examining failure patterns across facility and risk categories.

In [82]:
completed["Risk"].value_counts(dropna=False)

Risk
Risk 1 (High)      126233
Risk 2 (Medium)     33116
Risk 3 (Low)        12082
NaN                    34
All                     5
Name: count, dtype: int64

### 3. Risk Category Distribution

**Code:** `completed["Risk"].value_counts(dropna=False)`

**Result:** High-risk facilities account for 126,233 completed inspections, followed by medium-risk (33,116) and low-risk (12,082).

**Insight:** High-risk facilities dominate inspection records. Failure rates will be compared across risk categories to assess performance.

In [83]:
completed.groupby("Risk")["Results"].value_counts(normalize=True) * 100

Risk             Results           
All              Fail                  100.000000
Risk 1 (High)    Pass                   61.796044
                 Fail                   21.288411
                 Pass w/ Conditions     16.915545
Risk 2 (Medium)  Pass                   62.154246
                 Fail                   22.318517
                 Pass w/ Conditions     15.527238
Risk 3 (Low)     Pass                   61.256415
                 Fail                   31.153782
                 Pass w/ Conditions      7.589803
Name: proportion, dtype: float64

### 4. Failure Rates by Risk Category

**Code:**
```python
completed.groupby("Risk")["Results"].value_counts(normalize=True) * 100
```

**Result:** Low-risk facilities recorded the highest failure rate (31.15%), followed by medium-risk (22.32%) and high-risk (21.29%).

**Insight:** Inspection failure rates do not follow the expected risk ranking. Further analysis is needed to understand this pattern. The "All" category was excluded from meaningful comparisons due to its small sample size.

In [84]:
completed["Facility Type"].value_counts().head(10)

Facility Type
restaurant                         115674
grocery store                       22111
school                              11767
children's services facility         2941
bakery                               2530
daycare (2 - 6 years)                2385
daycare above and under 2 years      2248
long term care                       1292
catering                              996
mobile food dispenser                 801
Name: count, dtype: int64

### 5. Facility Type Distribution

**Code:** `completed["Facility Type"].value_counts().head(10)`

**Result:** Restaurants recorded the most completed inspections (115,674), followed by grocery stores (22,111) and schools (11,767).

**Insight:** Restaurants dominate inspection activity. Comparing failure rates will help identify which facility types have poorer inspection outcomes.

In [85]:
check = completed[completed["Facility Type"].isin(
    ["restaurant", "grocery store", "school"]
)]

check.groupby("Facility Type")["Results"].value_counts(normalize=True) * 100

Facility Type  Results           
grocery store  Pass                  59.757587
               Fail                  26.787572
               Pass w/ Conditions    13.454841
restaurant     Pass                  60.944551
               Fail                  21.138717
               Pass w/ Conditions    17.916732
school         Pass                  71.301096
               Fail                  20.234554
               Pass w/ Conditions     8.464349
Name: proportion, dtype: float64

### 6. Failure Rates by Facility Type

**Code:** `major.groupby("Facility Type")["Results"].value_counts(normalize=True) * 100`

**Result:** Grocery stores recorded the highest failure rate (26.79%), followed by restaurants (21.14%) and schools (20.23%).

**Insight:** Grocery stores had a higher failure rate despite restaurants accounting for more inspections.

In [86]:
completed["Inspection Date"].dt.year.value_counts().sort_index()

Inspection Date
2010    17382
2011    17584
2012    16018
2013    17068
2014    18621
2015    17865
2016    19838
2017    18480
2018    14805
2019    13809
Name: count, dtype: int64

### 7. Annual Inspection Trends

**Code:** `completed["Inspection Date"].dt.year.value_counts().sort_index()`

**Result:** Inspection activity peaked in 2016 (19,838). Among complete calendar years, 2018 recorded the fewest completed inspections (14,805).

**Insight:** Inspection volumes fluctuated over time, with a decline after 2017. The 2019 data covers only part of December.

In [87]:
year_groups = completed.groupby(completed["Inspection Date"].dt.year)["Results"]

year_groups.value_counts(normalize=True) * 100

Inspection Date  Results           
2010             Pass                  65.890001
                 Fail                  25.911863
                 Pass w/ Conditions     8.198136
2011             Pass                  66.611692
                 Fail                  24.766833
                 Pass w/ Conditions     8.621474
2012             Pass                  67.705082
                 Fail                  22.736921
                 Pass w/ Conditions     9.557997
2013             Pass                  70.096086
                 Fail                  19.621514
                 Pass w/ Conditions    10.282400
2014             Pass                  67.627947
                 Fail                  19.972075
                 Pass w/ Conditions    12.399979
2015             Pass                  67.774979
                 Fail                  20.162329
                 Pass w/ Conditions    12.062692
2016             Pass                  65.077125
                 Fail            

### 8. Annual Inspection Failure Rates

**Code:** `year_groups.value_counts(normalize=True) * 100`

**Result:** Failure rates declined from 25.91% in 2010 to 19.62% in 2013, then remained around 20–23% through 2019.

**Insight:** Conditional passes increased sharply in 2018–2019, requiring further investigation before drawing conclusions about inspection performance.

In [88]:
inspections_2018 = completed[completed["Inspection Date"].dt.year == 2018]

months = inspections_2018.groupby(inspections_2018["Inspection Date"].dt.month)["Results"]

months.value_counts(normalize=True) * 100

Inspection Date  Results           
1                Pass                  62.208068
                 Fail                  23.566879
                 Pass w/ Conditions    14.225053
2                Pass                  65.174129
                 Fail                  23.051410
                 Pass w/ Conditions    11.774461
3                Pass                  64.064801
                 Fail                  23.932253
                 Pass w/ Conditions    12.002946
4                Pass                  64.576384
                 Fail                  23.215477
                 Pass w/ Conditions    12.208139
5                Pass                  63.808855
                 Fail                  23.963457
                 Pass w/ Conditions    12.227688
6                Pass                  64.011586
                 Fail                  22.519913
                 Pass w/ Conditions    13.468501
7                Pass w/ Conditions    64.583333
                 Fail            

### 9. Investigating the 2018 Conditional Pass Increase

**Code:** `months.value_counts(normalize=True) * 100`

**Result:** Conditional passes increased sharply from 13.47% in June to 64.58% in July 2018, while unconditional passes declined.

**Insight:** The proportion of conditional passes increased sharply between June and July 2018. This indicates a significant change in recorded inspection outcomes, but the available analysis does not establish its cause.

In [89]:
months.value_counts()

Inspection Date  Results           
1                Pass                  879
                 Fail                  333
                 Pass w/ Conditions    201
2                Pass                  786
                 Fail                  278
                 Pass w/ Conditions    142
3                Pass                  870
                 Fail                  325
                 Pass w/ Conditions    163
4                Pass                  968
                 Fail                  348
                 Pass w/ Conditions    183
5                Pass                  908
                 Fail                  341
                 Pass w/ Conditions    174
6                Pass                  884
                 Fail                  311
                 Pass w/ Conditions    186
7                Pass w/ Conditions    651
                 Fail                  211
                 Pass                  146
8                Pass w/ Conditions    812
                 F

### 9. Investigating the 2018 Conditional Pass Increase — Counts

**Code:** `months.value_counts()`

**Result:** Conditional passes increased from 186 inspections in June to 651 in July 2018.

**Insight:** The increase in both counts and percentages confirms a substantial shift in reported inspection outcomes. The reason for this change remains unverified.

In [90]:
completed["Zip"].value_counts().head(10)

Zip
60614    6242
60647    6022
60657    5774
60611    5646
60622    5385
60608    5122
60618    4978
60607    4702
60639    4534
60640    4512
Name: count, dtype: int64[pyarrow]

### 10. Inspection Distribution by ZIP Code

**Code:** `completed["Zip"].value_counts().head(10)`

**Result:** ZIP codes 60614 (6,242), 60647 (6,022), and 60657 (5,774) recorded the highest numbers of completed inspections.

**Insight:** Inspection activity varies geographically. Failure rates are needed to assess performance across locations.

In [91]:
top_zips = completed["Zip"].value_counts().head(10).index

zip_data = completed[completed["Zip"].isin(top_zips)]

zip_data.groupby("Zip")["Results"].value_counts(normalize=True) * 100

Zip    Results           
60607  Pass                  60.867716
       Fail                  22.373458
       Pass w/ Conditions    16.758826
60608  Pass                  61.245607
       Fail                  23.291683
       Pass w/ Conditions    15.462710
60611  Pass                  64.895501
       Pass w/ Conditions    18.561814
       Fail                  16.542685
60614  Pass                  56.536367
       Fail                  22.685037
       Pass w/ Conditions    20.778597
60618  Pass                  67.336280
       Fail                  17.396545
       Pass w/ Conditions    15.267176
60622  Pass                  61.299907
       Fail                  23.639740
       Pass w/ Conditions    15.060353
60639  Pass                  59.307455
       Fail                  25.562417
       Pass w/ Conditions    15.130128
60640  Pass                  54.898050
       Fail                  26.484929
       Pass w/ Conditions    18.617021
60647  Pass                  59.315842

### 11. Geographic Failure Rates

**Code:** `zip_data.groupby("Zip")["Results"].value_counts(normalize=True) * 100`

**Result:** Among the 10 most-inspected ZIP codes, 60640 recorded the highest failure rate (26.48%), while 60611 recorded the lowest (16.54%).

**Insight:** Higher inspection volumes do not necessarily correspond to higher failure rates. Geographic differences warrant further exploration.

## Exploratory Data Analysis — Key Findings

**Dataset:** Chicago Food Inspections (2010–2019)

**Key Findings:**
1. **Overall Performance:** Of 171,470 completed inspections, 22.19% failed, 61.82% passed, and 15.99% passed with conditions.
2. **Risk Analysis:** Low-risk facilities recorded the highest observed failure rate (31.15%), compared with medium-risk (22.32%) and high-risk facilities (21.29%).
3. **Facility Performance:** Grocery stores recorded a higher failure rate (26.79%) than restaurants (21.14%) and schools (20.23%).
4. **Annual Trends:** Failure rates fluctuated over the period. Conditional passes increased sharply beginning in July 2018.
5. **Geographic Analysis:** ZIP code 60614 recorded the most completed inspections (6,242). Among the 10 most-inspected ZIP codes, 60640 had the highest failure rate (26.48%).

**Limitations:**
- Findings reflect historical inspection records, not current food safety conditions.
- Individual establishments may have been inspected multiple times.
- The cause of the July 2018 change in conditional passes remains unverified.
- Some missing and unresolved values were retained to avoid introducing inaccurate information.

**Conclusion:** Inspection outcomes vary across risk categories, facility types, time periods, and geographic areas. These findings will guide the Power BI dashboard.

In [92]:
import os
print(os.getcwd())

C:\Users\hp


In [93]:
import pandas as pd

df_small = pd.read_csv(
    'Chicago_Food_Inspections_Cleaned.csv',
    usecols=['Inspection ID', 'Violations'],
    nrows=1000
)

df_small.head()

,Inspection ID,Violations
0,2352683,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E..."
1,2352734,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...
2,2352713,NaN
3,2352701,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E..."
4,2352727,NaN


In [94]:
sample = df_small['Violations'].dropna().iloc[0]

violations = sample.split('|')

print(violations[:2])

['3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITIES AND REPORTING - Comments: FOUND NO EMPLOYEE HEALTH POLICY AT PREMISES. PRIORITY FOUNDATION. NO CITATION ISSUED. 7-38-010. MUST PROVIDE.']


In [95]:
code = violations[0].split('.')[0]

print(code)

3


In [96]:
description = violations[0].split(' - Comments:')[0]
description = description.split('. ', 1)[1]

print(description)

MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITIES AND REPORTING


In [97]:
codes = []

for text in df_small['Violations'].dropna():
    for violation in text.split('|'):
        code = violation.strip().split('.')[0]

        if code.isdigit():
            codes.append(code)

pd.Series(codes).value_counts().head(5)

55    618
5     291
3     265
47    249
51    225
Name: count, dtype: int64

In [98]:
codes = []

for text in df_clean['Violations'].dropna():
    for violation in text.split('|'):
        code = violation.strip().split('.')[0]

        if code.isdigit():
            codes.append(code)

top5 = pd.Series(codes).value_counts().head(5)

print(top5)

34    71758
35    64872
33    63728
38    57852
32    54946
Name: count, dtype: int64


In [99]:
for code in top5.index:
    match = next(
        (v.strip() for text in df_clean['Violations'].dropna()
         for v in text.split('|')
         if v.strip().startswith(code + '.')),
        None
    )

    if match:
        print(code, ':', match.split(' - Comments:')[0])

34 : 34. PLANT FOOD PROPERLY COOKED FOR HOT HOLDING
35 : 35. APPROVED THAWING METHODS USED
33 : 33. PROPER COOLING METHODS USED; ADEQUATE EQUIPMENT FOR TEMPERATURE CONTROL
38 : 38. INSECTS, RODENTS, & ANIMALS NOT PRESENT
32 : 32. VARIANCE OBTAINED FOR SPECIALIZED PROCESSING METHODS


In [100]:
descriptions = set()

for text in df_clean['Violations'].dropna():
    for violation in text.split('|'):
        if violation.strip().startswith('34.'):
            description = violation.split(' - Comments:')[0].strip()
            descriptions.add(description)

print(descriptions)

{'34. PLANT FOOD PROPERLY COOKED FOR HOT HOLDING', '34. FLOORS: CONSTRUCTED PER CODE, CLEANED, GOOD REPAIR, COVING INSTALLED, DUST-LESS CLEANING METHODS USED'}


In [101]:
violations_list = []

for text in df_clean['Violations'].dropna():
    for violation in text.split('|'):
        description = violation.split(' - Comments:')[0].strip()

        if description:
            violations_list.append(description)

top5 = pd.Series(violations_list).value_counts().head(5)

print(top5)

34. FLOORS: CONSTRUCTED PER CODE, CLEANED, GOOD REPAIR, COVING INSTALLED, DUST-LESS CLEANING METHODS USED                   71756
35. WALLS, CEILINGS, ATTACHED EQUIPMENT CONSTRUCTED PER CODE: GOOD REPAIR, SURFACES CLEAN AND DUST-LESS CLEANING METHODS    64747
33. FOOD AND NON-FOOD CONTACT EQUIPMENT UTENSILS CLEAN, FREE OF ABRASIVE DETERGENTS                                         62887
32. FOOD AND NON-FOOD CONTACT SURFACES PROPERLY DESIGNED, CONSTRUCTED AND MAINTAINED                                        54820
38. VENTILATION: ROOMS AND EQUIPMENT VENTED AS REQUIRED: PLUMBING: INSTALLED AND MAINTAINED                                 54165
Name: count, dtype: int64


In [102]:
top5.to_csv('Top_5_Violations.csv', header=['Count'])

## Phase 4: Top 5 Violations Analysis

**Objective:** Identify the five most frequently recorded food inspection violations.

**Method:** Separated violation descriptions, excluded inspector comments and counted how frequently each violation appeared.

**Finding:** The most common violations involved facility maintenance, equipment cleanliness, food-contact surfaces, ventilation and plumbing.

**Output:** Exported `Top_5_Violations.csv` for the Power BI dashboard.

**Note:** Violation frequencies are not unique inspection counts, as one inspection can contain multiple violations.

# Chicago Food Inspections: Data Cleaning and Exploratory Analysis Using Python

## 1. Project Overview

This project examines food inspection records from Chicago, with the aim of understanding inspection outcomes, failure rates, facility-level patterns, recurring violations and geographic differences.

The project combines Python for data preparation and analysis with Power BI for interactive reporting.

**Tools:** Python, pandas, Jupyter Notebook, Power BI Desktop

**Analysis period:** January 2010 – December 2019

**Final dataset:** 196,676 inspection records

## 2. Project Objectives

The analysis was designed to answer five questions:

1. How many inspections were conducted, and what were their outcomes?
2. What proportion of completed inspections resulted in failure?
3. How did failure rates change between 2010 and 2019?
4. Which facility types recorded the most failed inspections?
5. Which violations occurred most frequently, and how were failures distributed across Chicago ZIP codes?

## 3. Initial Data Assessment

The original dataset contained **196,825 rows and 22 columns**.

The initial review identified several issues requiring investigation before analysis:

- Missing values in some fields, including violation descriptions.
- Date values requiring conversion into a usable date format.
- Identifier fields requiring appropriate data-type handling.
- Geographic fields with incomplete information.
- Columns that were unnecessary for the intended analysis.

The recorded inspection dates ranged from January 4, 2010, to December 4, 2019.

**Documentation to retain from the existing notebook:** The original dataframe inspection outputs, missing-value summaries, data-type checks and any duplicate-record investigations.

## 4. Data Cleaning

Python was used to prepare the inspection dataset before importing it into Power BI.

### 4.1 Data Types

Date fields were prepared for time-based analysis, while identifiers were handled as identifiers rather than analytical quantities.

This was important because inspection records needed to support annual trends and accurate record-level counting.

### 4.2 Missing Values

Missing information was assessed according to its relevance to the analysis.

In particular, missing violation descriptions were treated differently from inspection counts. An inspection could still be included in overall outcome analysis even when its violation field was empty.

This avoided unnecessarily removing inspections simply because an optional descriptive field was missing.

### 4.3 Unnecessary Columns

Fields not required for the project's analysis were excluded from the cleaned dataset.

The original Community Area field was unavailable for usable geographic analysis, so the final geographic visualization was based on ZIP codes instead.

### 4.4 Final Dataset

The final cleaned dataset contained **196,676 inspection records**, compared with 196,825 initially.

This represents a reduction of 149 records.

The exact reasons for those 149 exclusions should be taken from the cleaning operations recorded in the notebook, rather than attributed to any single cleaning step without verification.

## 5. Exploratory Data Analysis

The exploratory analysis focused on the following areas.

### Inspection Outcomes

Inspection records were grouped by their reported results to examine the distribution of passing, failing and other outcomes.

For failure-rate calculations, completed inspections were defined as those with one of these outcomes:

- Pass
- Fail
- Pass w/ Conditions

Other outcomes remained part of total inspection counts but were excluded from the completed-inspection denominator.

### Inspection Failure Rates

The overall failure rate was calculated as:

**Failure Rate = Failed Inspections ÷ Completed Inspections × 100**

The final results were:

| Metric | Value |
|---|---:|
| Total inspections | 196,676 |
| Completed inspections | 171,470 |
| Failed inspections | 38,055 |
| Failure rate | 22.19% |

### Annual Trends

Inspection records were analyzed by year to identify changes in failure rates between 2010 and 2019.

The purpose was to examine changes in the proportion of failed inspections, rather than relying only on annual failure counts.

### Facility Types

Failed inspections were summarized by facility type.

The five facility types with the highest failed-inspection counts were:

| Facility type | Failed inspections |
|---|---:|
| Restaurant | 24,452 |
| Grocery store | 5,923 |
| School | 2,381 |
| Bakery | 621 |
| Children's services facility | 548 |

These figures represent counts of failed inspections, not facility-specific failure rates.

### Violation Analysis

Violation descriptions were analyzed to identify frequently reported issues.

A separate overall Top 5 Violations summary was prepared for Power BI.

The final summary counts were:

| Rank | Count |
|---|---:|
| 1 | 71,756 |
| 2 | 64,747 |
| 3 | 62,887 |
| 4 | 54,820 |
| 5 | 54,165 |

The corresponding violation labels are maintained in the exported summary table. Counts of violations should not be interpreted as counts of unique inspections because an inspection can have multiple violations.

## 6. Preparing Data for Power BI

The cleaned inspection dataset was exported for reporting in Power BI.

A separate summary containing the five most frequently identified violations was also prepared.

The two datasets served different purposes:

- **Cleaned inspections data:** Used for interactive KPI measures, outcome analysis, trends, facility comparisons and ZIP code mapping.
- **Top 5 Violations summary:** Used as a static overall reference for the 2010–2019 period.

## 7. Conclusion

The Python stage produced a cleaned inspection dataset suitable for interactive analysis and a separate summary of recurring violations.

The results provided the foundation for the Power BI dashboard, where inspection outcomes, annual trends, facility categories and geographic concentrations could be explored visually.

In [103]:
df.shape

(196825, 22)

In [104]:
from pathlib import Path
import shutil

source = Path(path) / "food-inspections.csv"
destination = Path.cwd() / "Chicago_Food_Inspections_Raw.csv"

shutil.copy2(source, destination)
print(destination)

C:\Users\hp\Chicago_Food_Inspections_Raw.csv
